# 00 — Business problem and PoC design

## Context

Healthcare networks plan staff weeks in advance. Each clinic needs a roster of
clinicians, nurses and front-desk staff sized to the demand it will actually
face. Two failure modes carry real cost:

- **Understaffing** lengthens waiting times, forces overtime (typically paid at
  a 1.5x premium), burns out staff and degrades patient experience.
- **Overstaffing** wastes salary budget on idle capacity that cannot be
  recovered.

Both failure modes share a root cause: rosters are usually built from rules of
thumb and last year's averages, not from a forecast of demand. This proof of
concept tests whether a forecasting system at **clinic x day** granularity,
with a **28-day planning horizon**, can support materially better staffing
decisions.

## Decisions this system supports

| Decision | Who makes it | Cadence | What the forecast contributes |
|---|---|---|---|
| Base roster per clinic | Regional operations manager | Monthly | Expected daily visits, 28 days ahead |
| Overtime / float pool allocation | Clinic manager | Weekly | Upper-bound (conservative) forecast |
| Marketing coordination | Marketing + operations | Campaign-by-campaign | Demand uplift under spend scenarios |
| Capacity review (open Saturdays, extend hours) | Network leadership | Quarterly | Utilisation pressure trends |

The forecast is decision support, not automation: staffing recommendations are
reviewed by operations managers before rosters change. This matters in
healthcare, where the cost of forecast errors is asymmetric and context the
model cannot see (local events, staff illness) is common.

## Data entities

The PoC uses synthetic aggregate data with four main tables, written to
`data/processed/` by the data generator.

| Table | Grain | Description |
|---|---:|---|
| `clinic_daily_usage` | clinic x day | scheduled appointments, visits, no-shows, cancellations, utilisation |
| `clinic_metadata` | clinic | region, size, specialty, capacity, baseline staff, weekend opening |
| `marketing_daily` | clinic x day | campaign flag and spend by channel (search, social, email, local) |
| `staffing_daily` | clinic x day | staffed clinicians, nurses and front-desk per day |

No patient-level fields are generated or required. The simulation deliberately
includes the properties that make real clinic demand hard to forecast:
overdispersed counts, persistent demand episodes (e.g. flu waves), trend
changepoints, structural zeros from closures, and marketing effects with
carryover and diminishing returns. Notebook 01 documents the design.

## Forecast targets and the schedule/realised distinction

The primary target is **completed visits** per clinic per day. The data also
distinguishes **scheduled appointments**, which exceed visits by the no-show
and same-day cancellation share. The distinction is operationally important:

- Rooms and front-desk load follow the *schedule*.
- Clinical workload and revenue follow *completed visits*.
- The gap between them (no-shows) is itself forecastable and actionable
  (reminder campaigns, overbooking policies).

The PoC forecasts visits first and treats no-show forecasting as a planned
extension (see `ROADMAP.md`).

## Evaluation design

- **Rolling-origin backtesting** (`RollingOriginSplitter`): models are
  evaluated on multiple 28-day windows, never on a single lucky split, and
  training data always ends strictly before the test window.
- **Primary metric: WAPE** (weighted absolute percentage error) — robust to
  the zero-visit days that closures create, and weights errors by volume,
  which matches staffing cost. MAPE is reported but not trusted: it is
  undefined on zero days and explodes on small ones.
- **Bias** is tracked separately because staffing penalties are asymmetric:
  systematic under-forecasting causes understaffing, which costs more than the
  equivalent overstaffing.
- **The bar is the seasonal naive baseline.** Any model that cannot clearly
  beat "repeat last week" does not justify its complexity.

## Scope and non-goals

**In scope:** multi-clinic demand forecasting (baselines, SARIMAX, global ML),
rolling-origin validation, prediction intervals (planned), staffing decision
layer, batch outputs and a small serving API.

**Out of scope, deliberately:** patient-level modelling, clinical outcome
prediction, real-time streaming, automated roster changes without human
review, and causal claims about marketing effectiveness (scenario analysis is
model-based what-if, not causal inference).

## Notebook map

| Notebook | Question it answers |
|---|---|
| 01 | What data are we working with, and why is it shaped this way? |
| 02 | What patterns drive demand, and how forecastable is it? |
| 03 | How far do classical statistical models get us? |
| 04 | Does one global ML model beat per-clinic approaches at scale? |
| 05 | What would deep-learning / foundation-model benchmarks add? |
| 06 | How do forecasts become staffing decisions and costs? |